In [2]:
import os
import pandas as pd
import statsmodels.api as sm
from statsmodels.tsa.ar_model import AutoReg
from sklearn.utils import resample
import numpy as np

os.chdir('/Users/fogellmcmuffin/Documents/thesis/_replication/')    # Working dir

pd.set_option('display.max_rows', 10)
pd.set_option('display.max_columns', None)

vintage_trim = pd.read_csv('cleaned_data/vintage_trim.csv')

mean_spf_trim = pd.read_csv('cleaned_data/mean_spf_trim.csv')
ind_spf_trim = pd.read_csv('cleaned_data/ind_spf_trim.csv')
mean_spf_trim_nr = pd.read_csv('cleaned_data/mean_spf_trim_nr.csv')
ind_spf_trim_nr = pd.read_csv('cleaned_data/ind_spf_trim_nr.csv')

mean_spf_trim['DATE'] = pd.to_datetime(mean_spf_trim['DATE'])
mean_spf_trim_nr['DATE'] = pd.to_datetime(mean_spf_trim_nr['DATE'])
ind_spf_trim['DATE'] = pd.to_datetime(ind_spf_trim['DATE'])
ind_spf_trim_nr['DATE'] = pd.to_datetime(ind_spf_trim_nr['DATE'])

ind_spf_trim = ind_spf_trim.dropna(subset='r_t1')
ind_spf_trim_nr = ind_spf_trim_nr.dropna(subset='r_t1')

###############################################
                ## ESTIMATION ##
###############################################

### OLS ###
def OLS(df, end_date):
    df = df.loc[(df['DATE'] >= '1981-12-31') & (df['DATE'] <= end_date)]
    regs = []
    for j in range(4):
        revisions = df[f'r_t{j}']
        revisions = sm.add_constant(revisions)
        errors = df[f'e_t{j}']
        initial = sm.OLS(errors, revisions).fit()
        regs.append(initial.get_robustcov_results(cov_type='HAC', maxlags=None))
    return regs


### CG Estimation ###
def cg(df, end_date):
    df = df.loc[(df['DATE'] >= '1981-12-31') & (df['DATE'] <= end_date)]
    revisions = df[f'r_t30']
    revisions = sm.add_constant(revisions)
    errors = df[f'e_t30']
    initial = sm.OLS(errors, revisions).fit()
    regs = initial.get_robustcov_results(cov_type='HAC', maxlags=None)
    return regs


### ID FE ###
def ID_FE(df, end_date):
    df = df.loc[(df['DATE'] >= '1981-12-31') & (df['DATE'] <= end_date)]
    regs = []
    for j in range(4):
        x = np.column_stack((np.ones(len(df)), df[f'r_t{j}'], pd.get_dummies(df['ID'], drop_first=True, dtype=float)))
        y = np.asarray(df[f'e_t{j}'])
        initial = sm.OLS(y, x).fit()
        regs.append(initial.get_robustcov_results(cov_type='HAC', maxlags=None))
    return regs


### Two-way Fixed Effects ###
def T_FE(df, end_date):
    df = df.loc[(df['DATE'] >= '1981-12-31') & (df['DATE'] <= end_date)]
    regs = []
    for j in range(4):
        x = np.column_stack((np.ones(len(df)), df[f'r_t{j}'], pd.get_dummies(df['DATE'], drop_first=True, dtype=float)))
        y = np.asarray(df[f'e_t{j}'])
        initial = sm.OLS(y, x).fit()
        regs.append(initial.get_robustcov_results(cov_type='HC2'))
    return regs


### AR Estimates ###
def AR(df, end_date):
    df = df.loc[(df['DATE'] >= '1965-06-30') & (df['DATE'] <= end_date)]  # Filter data
    grwth_t = df['t3']
    reg = AutoReg(grwth_t, 1).fit()
    return reg


### Parameter Estimates ###
def params(ols, pldols, fe, fe2, ar1):
    params = []
    params.append((1+ols) / (1+(1+ols)))
    params.append(1 / (1 + (1+ols)))
    params.append((-((2 * pldols) + 1) + np.sqrt(((2 * pldols) + 1)**2 - 4 * (pldols + (pldols * ar1**2) + 1) * pldols)) / (2 * (pldols + (pldols * ar1**2) + 1)))
    params.append((-((2 * fe) + 1) + np.sqrt(((2 * fe) + 1)**2 - 4 * (fe + (fe * ar1**2) + 1) * fe)) / (2 * (fe + (fe * ar1**2) + 1)))
    params.append((-((2 * fe2) + 1) + np.sqrt(((2 * fe2) + 1)**2 - 4 * (fe2 + (fe2 * ar1**2) + 1) * fe2)) / (2 * (fe2 + (fe2 * ar1**2) + 1)))
    return params


def compute_regs(date, mean, ind):
    mean_regs = OLS(mean, f'{date}')
    cg_regs = cg(mean, f'{date}')
    ind_regs = OLS(ind, f'{date}')
    ind_regs_fe = ID_FE(ind, f'{date}')
    ind_regs_fe2 = T_FE(ind, f'{date}')
    ar_1 = AR(vintage_trim, f'{date}')
    regs = [mean_regs, ind_regs, ind_regs_fe, ind_regs_fe2, ar_1, cg_regs]
    parameters = params(mean_regs[3].params[1], ind_regs[3].params[1], ind_regs_fe[3].params[1], ind_regs_fe2[3].params[1], ar_1.roots)
    return regs, parameters

regs_2020, parameters_2020 = compute_regs('2020-06-30', mean_spf_trim, ind_spf_trim)
regs_2020_nr, parameters_2020_nr = compute_regs('2020-06-30', mean_spf_trim_nr, ind_spf_trim_nr)
%store regs_2020
%store parameters_2020
%store regs_2020_nr
%store parameters_2020_nr

regs_2022, parameters_2022 = compute_regs('2022-12-31', mean_spf_trim, ind_spf_trim)
regs_2022_nr, parameters_2022_nr = compute_regs('2022-12-31', mean_spf_trim_nr, ind_spf_trim_nr)
%store regs_2022
%store parameters_2022
%store regs_2022_nr
%store parameters_2022_nr

/Users/fogellmcmuffin/anaconda3/lib/python3.11/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: An unsupported index was provided and will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
/var/folders/6q/zww1c4xj04d9c7jlxyz1zc6c0000gn/T/ipykernel_56796/4019764422.py:95: RuntimeWarning: invalid value encountered in sqrt
  params.append((-((2 * fe2) + 1) + np.sqrt(((2 * fe2) + 1)**2 - 4 * (fe2 + (fe2 * ar1**2) + 1) * fe2)) / (2 * (fe2 + (fe2 * ar1**2) + 1)))


KeyError: 'r_t30'

In [59]:
regs_2020[0][3].summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                   e_t3   R-squared:                       0.007
Model:                            OLS   Adj. R-squared:                  0.001
Method:                 Least Squares   F-statistic:                     1.102
Date:                Sat, 27 Apr 2024   Prob (F-statistic):              0.296
Time:                        16:57:12   Log-Likelihood:                 486.07
No. Observations:                 155   AIC:                            -968.1
Df Residuals:                     153   BIC:                            -962.1
Df Model:                           1                                         
Covariance Type:                  HAC                                         
==============================================================================
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const         -0.0024      0.001     -1.772      0.078      -0.005       0.000
r_t3           0.1939      0.185      1.050      0.296      -0.171       0.559
==============================================================================
Omnibus:                       11.186   Durbin-Watson:                   0.625
Prob(Omnibus):                  0.004   Jarque-Bera (JB):               16.749
Skew:                          -0.391   Prob(JB):                     0.000231
Kurtosis:                       4.408   Cond. No.                         219.
==============================================================================

Notes:
[1] Standard Errors are heteroscedasticity and autocorrelation robust (HAC) using None lags and without small sample correction
"""

In [6]:
###############################################
              ## ID BOOTSTRAP ##
###############################################
def boot(df, R, date):
    RR = R + 1
    boot_regs0 = []
    boot_regs1 = []
    boot_regs2 = []
    boot_regs3 = []
    for i in range(1, RR):
        boot = resample(df['ID'], replace=True, n_samples=len(df), random_state=i)
        boot_df = df[df['ID'].isin(boot)]
        boot_ols = OLS(boot_df, date)
        boot_regs0.append(boot_ols[0].params[1])
        boot_regs1.append(boot_ols[1].params[1])
        boot_regs2.append(boot_ols[2].params[1])
        boot_regs3.append(boot_ols[3].params[1])
    boot_regs = [boot_regs0, boot_regs1, boot_regs2, boot_regs3]
    return boot_regs

boot_regs_2016 = boot(ind_spf_trim, 10000, '2016-12-31')
%store boot_regs_2016

boot_regs_2019 = boot(ind_spf_trim, 10000, '2019-12-31')
%store boot_regs_2019

boot_regs_2022 = boot(ind_spf_trim, 10000, '2022-12-31')
%store boot_regs_2022

boot_regs_2022n = boot(ind_spf_trim22, 10000, '2022-12-31')
%store boot_regs_2022n

Stored 'boot_regs_2016' (list)
Stored 'boot_regs_2019' (list)
Stored 'boot_regs_2022' (list)
Stored 'boot_regs_2022n' (list)


In [7]:
%store -r boot_regs_2016
%store -r boot_regs_2019
%store -r boot_regs_2022
%store -r boot_regs_2022n
print(np.quantile(boot_regs_2016[3], [0.025, 0.975]))
print(np.quantile(boot_regs_2016[3], 0.5))
print(np.quantile(boot_regs_2019[3], [0.025, 0.975]))
print(np.quantile(boot_regs_2019[3], 0.5))
print(np.quantile(boot_regs_2022[3], [0.025, 0.975]))
print(np.quantile(boot_regs_2022[3], 0.5))
print(np.quantile(boot_regs_2022n[3], [0.025, 0.975]))
print(np.quantile(boot_regs_2022n[3], 0.5))

[-0.19400186 -0.19146638]
-0.19176521394011467
[-0.20467117 -0.20218268]
-0.20245905135330092
[0.0602634 0.0625437]
0.062246529362919856
[-0.13942712 -0.13702902]
-0.1373404364100622


In [4]:
###############################################
    ## FORECASTER-BY-FORECASTER BOOTSTRAP ##
###############################################
